Source: Cricsheet ball-by-ball data for men's Tests, ODIs and T20Is. Output: `bangladesh_batting_innings.csv`, `pool_selection.csv`.

In [1]:
import io
import json
import zipfile
from collections import Counter

import pandas as pd
import requests

DOWNLOAD_URL = "https://cricsheet.org/downloads/"
FORMAT_ARCHIVES = {
    "Test": "tests_male_json.zip",
    "ODI": "odis_male_json.zip",
    "T20I": "t20s_male_json.zip",
}
TEAM = "Bangladesh"
SENA = {"South Africa", "England", "New Zealand", "Australia"}
MAJOR = {"India", "Pakistan", "Sri Lanka", "West Indies"}
UAE_CITIES = {"Dubai", "Sharjah", "Abu Dhabi"}
NOT_DISMISSALS = {"retired hurt", "retired not out"}
STAT_COLUMNS = ["Runs", "Balls", "Fours", "Sixes", "Dots", "Dismissed"]

In [2]:
def read_matches(archive_name):
    response = requests.get(DOWNLOAD_URL + archive_name, timeout=300)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        for member in archive.namelist():
            if member.endswith(".json"):
                yield member.removesuffix(".json"), json.loads(archive.read(member))


def match_result(outcome):
    winner = outcome.get("winner")
    if winner is None:
        return outcome.get("result", "no result").title()
    return "Won" if winner == TEAM else "Lost"


def batting_stats(innings):
    batters = {}
    for over in innings.get("overs", []):
        for ball in over["deliveries"]:
            batters.setdefault(ball["non_striker"], Counter())
            batter = batters.setdefault(ball["batter"], Counter())
            runs = ball["runs"]
            if "wides" not in ball.get("extras", {}):
                batter["Balls"] += 1
                batter["Runs"] += runs["batter"]
                batter["Dots"] += runs["batter"] == 0
                if not runs.get("non_boundary"):
                    batter["Fours"] += runs["batter"] == 4
                    batter["Sixes"] += runs["batter"] == 6
            for wicket in ball.get("wickets", []):
                if wicket["kind"] not in NOT_DISMISSALS:
                    batters.setdefault(wicket["player_out"], Counter())["Dismissed"] = 1
    return batters


def innings_rows(match_id, match, format_name):
    info = match["info"]
    people = info["registry"]["people"]
    context = {
        "Match_ID": match_id,
        "Date": info["dates"][0],
        "Format": format_name,
        "Opponent": next(team for team in info["teams"] if team != TEAM),
        "Venue": info.get("venue", "Unknown"),
        "City": info.get("city", ""),
        "Result": match_result(info["outcome"]),
    }
    rows = []
    for innings_no, innings in enumerate(match["innings"], start=1):
        if innings["team"] != TEAM or innings.get("super_over"):
            continue
        for name, stats in batting_stats(innings).items():
            rows.append({
                **context,
                "Innings_No": innings_no,
                "Player_ID": people[name],
                "Player": name,
                **{column: stats[column] for column in STAT_COLUMNS},
            })
    return rows


def opponent_group(team):
    if team in SENA:
        return "SENA"
    return "Major" if team in MAJOR else "Minor"


def venue_type(host, opponent, city):
    if host == TEAM:
        return "Home"
    if city in UAE_CITIES:
        return "Neutral"
    return "Away" if host == opponent else "Neutral"

In [3]:
venue_teams = {}
rows = []
for format_name, archive_name in FORMAT_ARCHIVES.items():
    for match_id, match in read_matches(archive_name):
        info = match["info"]
        venue_teams.setdefault(info.get("venue", "Unknown"), Counter()).update(info["teams"])
        if TEAM in info["teams"]:
            rows.extend(innings_rows(match_id, match, format_name))

innings = pd.DataFrame(rows)
len(innings)

7290

In [4]:
venue_host = {venue: teams.most_common(1)[0][0] for venue, teams in venue_teams.items()}

innings["Opponent_Group"] = innings["Opponent"].map(opponent_group)
innings["Venue_Type"] = [
    venue_type(venue_host[venue], opponent, city)
    for venue, opponent, city in zip(innings["Venue"], innings["Opponent"], innings["City"])
]
innings["Boundary_Runs"] = 4 * innings["Fours"] + 6 * innings["Sixes"]
innings["Fifties"] = innings["Runs"].between(50, 99).astype(int)
innings["Hundreds"] = (innings["Runs"] >= 100).astype(int)

names = innings.groupby("Player_ID")["Player"].agg(lambda s: s.mode().iat[0])
innings["Player"] = innings["Player_ID"].map(names)

In [5]:
matches = innings.drop_duplicates("Match_ID")
display(matches.groupby("Format").agg(Matches=("Match_ID", "count"), First=("Date", "min"), Last=("Date", "max")))
display(matches.groupby(["Format", "Venue_Type"]).size().unstack(fill_value=0))
display(matches.loc[matches["Venue_Type"] == "Neutral", ["Venue", "Opponent"]].drop_duplicates())

,Matches,First,Last
Format,,,
ODI,330,2003-02-11,2026-10-17
T20I,201,2007-09-01,2026-10-03
Test,126,2003-07-18,2026-10-28


Venue_Type,Away,Home,Neutral
Format,,,
ODI,117,155,58
T20I,66,83,52
Test,58,68,0


,Venue,Opponent
2535,"Sheikh Zayed Stadium, Abu Dhabi",Zimbabwe
2587,The Oval,Australia
2598,"Sophia Gardens, Cardiff",New Zealand
2605,Edgbaston,India
2649,"Clontarf Cricket Club Ground, Dublin",New Zealand
...,...,...
7165,"Himachal Pradesh Cricket Association Stadium, ...",Netherlands
7174,"Himachal Pradesh Cricket Association Stadium, ...",Ireland
7177,"Himachal Pradesh Cricket Association Stadium, ...",Oman
7189,"M Chinnaswamy Stadium, Bengaluru",Australia


In [6]:
POOL_SIZE = 10

ranking = innings.pivot_table(index="Player_ID", columns="Format", values="Runs", aggfunc="sum", fill_value=0)
ranking["Total_Runs"] = ranking.sum(axis=1)
ranking = ranking.sort_values("Total_Runs", ascending=False)
ranking.columns.name = None
ranking.insert(0, "Player", ranking.index.map(names))
ranking.insert(0, "Rank", range(1, len(ranking) + 1))
ranking["Run_Share_Pct"] = (100 * ranking["Total_Runs"] / ranking["Total_Runs"].sum()).round(2)
ranking["Cumulative_Share_Pct"] = ranking["Run_Share_Pct"].cumsum().round(2)
ranking["In_Pool"] = ranking["Rank"] <= POOL_SIZE

pool_runs = ranking.loc[ranking["In_Pool"], "Total_Runs"].sum()
print(f"Top {POOL_SIZE} of {len(ranking)} batters ({POOL_SIZE / len(ranking):.1%}) scored {pool_runs / ranking['Total_Runs'].sum():.1%} of all runs")
display(ranking.head(20))

Top 10 of 129 batters (7.8%) scored 56.8% of all runs


,Rank,Player,ODI,T20I,Test,Total_Runs,Run_Share_Pct,Cumulative_Share_Pct,In_Pool
Player_ID,,,,,,,,,
a94e08ea,1,Mushfiqur Rahim,7096,1307,6592,14995,10.05,10.05,True
3b041a12,2,Tamim Iqbal,7648,1567,5019,14234,9.54,19.59,True
7dc35884,3,Shakib Al Hasan,6638,2295,4356,13289,8.91,28.50,True
bf1d7d3e,4,Mahmudullah,5131,2223,2900,10254,6.87,35.37,True
0404d43c,5,Liton Das,2478,2469,3277,8224,5.51,40.88,True
de0a3209,6,Mominul Haque,495,60,5056,5611,3.76,44.64,True
1274d7ab,7,Nazmul Hossain Shanto,1746,959,2453,5158,3.46,48.10,True
4d9f9686,8,Soumya Sarkar,2343,1466,799,4608,3.09,51.19,True
2bb09eb2,9,Mohammad Ashraful,2301,382,1523,4206,2.82,54.01,True


In [7]:
from google.colab import files

pool_ids = ranking.index[ranking["In_Pool"]]
innings[innings["Player_ID"].isin(pool_ids)].to_csv("bangladesh_batting_innings.csv", index=False)
ranking.reset_index().to_csv("pool_selection.csv", index=False)

files.download("bangladesh_batting_innings.csv")
files.download("pool_selection.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>